# Exploratory Data Analysis (EDA)
## Understanding the Silver.Sales Table

### Purpose of This Notebook
Before performing any analysis or building reports, we must **thoroughly understand our data**. This notebook demonstrates a systematic approach to exploring a new dataset.

### What We'll Explore
1. **Table Overview** - Basic statistics, row counts, time coverage
2. **Dimensional Coverage** - How many stores, departments, categories, products?
3. **Geographic Distribution** - Regions, states, cities in our data
4. **Time Period Analysis** - Date ranges overall and by store
5. **Data Quality** - Missing values, duplicates, data integrity
6. **Value Ranges** - Min/max for quantities, amounts, discounts
7. **Distribution Patterns** - Transaction patterns, basket sizes

### Why This Matters
- **Data Quality**: Identify issues before they affect analysis
- **Context**: Understand what the data represents
- **Scope**: Know the boundaries and limitations
- **Planning**: Make informed decisions about next steps

---
*Note: This is exploratory work - we're NOT calculating business metrics yet. That comes later.*

## 1. Table Overview

### What We're Checking:
- Total number of records (rows)
- Number of columns
- Overall time period covered by the data
- Earliest and latest transaction dates

### Why This Matters:
- **Row count** tells us the volume of data we're working with
- **Time period** helps us understand if we have complete coverage or just a sample
- **Date range** is critical for time-based analysis later

### Expected:
We should see millions of line items spanning several months of retail transactions.

In [0]:
-- Basic table statistics
-- This query gives us the foundation: how much data do we have?

SELECT 
  COUNT(*) AS total_records,
  COUNT(DISTINCT transaction_id) AS unique_transactions,
  COUNT(DISTINCT line_item_id) AS unique_line_items,
  MIN(transaction_date) AS earliest_date,
  MAX(transaction_date) AS latest_date,
  DATEDIFF(MAX(transaction_date), MIN(transaction_date)) AS days_covered
FROM workspace.silver.sales

total_records,unique_transactions,unique_line_items,earliest_date,latest_date,days_covered
4846789,1375105,4846789,2025-01-01,2026-05-31,515


## 2. Dimensional Coverage

### What We're Checking:
- How many unique values in each dimension?
  - Stores
  - Departments
  - Categories
  - Products
  - SKUs
  - Customers

### Why This Matters:
- Tells us the **breadth** of our data
- Helps identify if we have a complete product catalog or subset
- Understanding customer base size
- Validates that dimension tables were properly joined

### What to Look For:
- Do counts match our expectations from dimension tables?
- Are there reasonable ratios? (e.g., multiple SKUs per product)

In [0]:
-- Count unique values in each dimension
-- This shows how many distinct entities we're tracking

SELECT dimension, unique_count
FROM (
  SELECT 
    COUNT(DISTINCT customer_id) AS Customers,
    COUNT(DISTINCT sku_id)      AS SKUs,
    COUNT(DISTINCT product_id)  AS Products,
    COUNT(DISTINCT category_id)    AS Categories,
    COUNT(DISTINCT store_id)    AS Stores,
    COUNT(DISTINCT department_id)  AS Departments
  FROM workspace.silver.sales
)
UNPIVOT (
  unique_count FOR dimension IN (Customers, SKUs, Products, Categories, Stores, Departments)
)
ORDER BY unique_count DESC;

ORDER BY unique_count DESC

dimension,unique_count
Customers,14359
SKUs,1418
Products,591
Categories,36
Stores,21
Departments,7


## 3. Material Hierarchy Deep Dive

### What We're Checking:
- Department-wise breakdown (how many categories per department?)
- Category-wise breakdown (how many products per category?)
- Product-wise breakdown (how many SKUs per product?)

### Why This Matters:
- Understand the **structure** of our product catalog
- Identify if some departments/categories are more granular than others
- Spot potential data quality issues (e.g., products with no SKUs)

### What to Look For:
- Balanced hierarchy or skewed?
- Any departments dominating the catalog?

In [0]:
-- Department-level summary
-- Shows how our product catalog is organized at the highest level

SELECT 
  department_id,
  department_name,
  COUNT(DISTINCT category_id) AS num_categories,
  COUNT(DISTINCT product_id) AS num_products,
  COUNT(DISTINCT sku_id) AS num_skus,
  COUNT(*) AS num_line_items
FROM workspace.silver.sales
GROUP BY department_id, department_name
ORDER BY num_line_items DESC

department_id,department_name,num_categories,num_products,num_skus,num_line_items
DEPT-01,Grocery & Staples,7,126,318,1440541
DEPT-02,Dairy & Frozen,5,80,190,1265133
DEPT-05,Beverages,5,90,248,1232817
DEPT-04,Household,5,75,178,478002
DEPT-03,Personal Care,5,80,199,222931
DEPT-07,Fashion & Lifestyle,5,80,160,121880
DEPT-06,Electronics & Accessories,4,60,125,85485


In [0]:
-- Top 15 categories by transaction volume
-- Helps identify which product categories drive the most activity

SELECT 
  department_name,
  category_name,
  COUNT(DISTINCT product_id) AS num_products,
  COUNT(DISTINCT sku_id) AS num_skus,
  COUNT(*) AS num_line_items,
  COUNT(DISTINCT transaction_id) AS num_transactions
FROM workspace.silver.sales
GROUP BY department_name, category_name
ORDER BY num_line_items DESC
LIMIT 15

department_name,category_name,num_products,num_skus,num_line_items,num_transactions
Dairy & Frozen,Milk & Curd,16,36,767269,593280
Beverages,Tea & Coffee,18,59,520947,423349
Grocery & Staples,Flour & Atta,18,46,353156,302937
Beverages,Soft Drinks & Juices,18,51,323422,288244
Grocery & Staples,Sugar & Salt,18,42,257435,235946
Household,Cleaning Supplies,15,38,235941,189490
Grocery & Staples,Rice & Grains,18,47,226196,203492
Grocery & Staples,Cooking Oil & Ghee,18,48,220576,199538
Grocery & Staples,Pulses & Lentils,18,45,203575,187582
Dairy & Frozen,Frozen Snacks & Ready Meals,16,38,183751,164206


## 4. Geographic Distribution

### What We're Checking:
- How many regions, states, and cities?
- Store distribution across geography
- Transaction volume by region

### Why This Matters:
- Understand **geographic coverage** of our retail chain
- Identify regional concentration or spread
- Later analysis may need regional comparisons

### What to Look For:
- Is the business concentrated in certain regions?
- Do all regions have similar store counts?
- Any geographic gaps?

In [0]:
-- Geographic hierarchy: Regions → States → Cities → Stores
-- This shows how our retail footprint is distributed

SELECT 
  region,
  COUNT(DISTINCT state) AS num_states,
  COUNT(DISTINCT city) AS num_cities,
  COUNT(DISTINCT store_id) AS num_stores,
  COUNT(*) AS num_line_items,
  COUNT(DISTINCT transaction_id) AS num_transactions
FROM workspace.silver.sales
GROUP BY region
ORDER BY num_line_items DESC

region,num_states,num_cities,num_stores,num_line_items,num_transactions
North,4,5,5,1245333,354832
West,2,4,4,968580,268601
South,3,4,4,958279,266342
East,4,4,4,926546,260355
Central,2,4,4,748051,224975


In [0]:
-- Complete store list with location and format
-- Useful reference to see all stores in our dataset

SELECT DISTINCT
  store_id,
  store_name,
  city,
  state,
  region,
  tier,
  format
FROM workspace.silver.sales
ORDER BY region, state, city, store_name

store_id,store_name,city,state,region,tier,format
STR-RPR-001,FreshMart Raipur,Raipur,Chhattisgarh,Central,3,Express
STR-BPL-001,FreshMart Bhopal,Bhopal,Madhya Pradesh,Central,2,Supermarket
STR-IND-001,FreshMart Indore,Indore,Madhya Pradesh,Central,2,Fresh
STR-JBP-001,FreshMart Jabalpur,Jabalpur,Madhya Pradesh,Central,3,Express
STR-GUW-001,FreshMart Guwahati,Guwahati,Assam,East,2,Supermarket
STR-PAT-001,FreshMart Patna,Patna,Bihar,East,3,Express
STR-BBS-001,FreshMart Bhubaneswar,Bhubaneswar,Odisha,East,2,Fresh
STR-KOL-001,FreshMart Kolkata,Kolkata,West Bengal,East,1,Supermarket
STR-CHD-001,FreshMart Chandigarh,Chandigarh,Punjab,North,1,Supermarket
STR-JAI-001,FreshMart Jaipur,Jaipur,Rajasthan,North,2,Supermarket


## 5. Time Period Analysis

### What We're Checking:
- Date range for each store
- Daily transaction counts
- Any gaps in data (missing dates)?
- Consistency across stores

### Why This Matters:
- Some stores may have **incomplete data**
- New stores may have opened mid-period
- Closed stores may have stopped reporting
- Gaps could indicate data loading issues

### What to Look For:
- All stores operating for the full period?
- Any unusual gaps or patterns?
- Consistent daily activity?

In [0]:
-- Date range per store
-- Identifies if any stores have partial data

SELECT 
  store_id,
  store_name,
  city,
  MIN(transaction_date) AS first_transaction,
  MAX(transaction_date) AS last_transaction,
  DATEDIFF(MAX(transaction_date), MIN(transaction_date)) AS days_active,
  COUNT(DISTINCT transaction_date) AS distinct_days_with_sales,
  COUNT(*) AS total_line_items
FROM workspace.silver.sales
GROUP BY store_id, store_name, city
ORDER BY first_transaction, store_name LIMIT 20;

store_id,store_name,city,first_transaction,last_transaction,days_active,distinct_days_with_sales,total_line_items
STR-BLR-001,FreshMart Bangalore,Bangalore,2025-01-01,2026-05-31,515,516,341686
STR-BPL-001,FreshMart Bhopal,Bhopal,2025-01-01,2026-05-31,515,516,274902
STR-BBS-001,FreshMart Bhubaneswar,Bhubaneswar,2025-01-01,2026-05-31,515,516,166252
STR-CHD-001,FreshMart Chandigarh,Chandigarh,2025-01-01,2026-05-31,515,516,313070
STR-COI-001,FreshMart Coimbatore,Coimbatore,2025-01-01,2026-05-31,515,516,182989
STR-DHN-001,FreshMart Dehradun,Dehradun,2025-01-01,2026-05-31,515,516,147979
STR-GUW-001,FreshMart Guwahati,Guwahati,2025-01-01,2026-05-31,515,516,270728
STR-IND-001,FreshMart Indore,Indore,2025-01-01,2026-05-31,515,516,198485
STR-JBP-001,FreshMart Jabalpur,Jabalpur,2025-01-01,2026-05-31,515,516,131201
STR-JAI-001,FreshMart Jaipur,Jaipur,2025-01-01,2026-05-31,515,516,291344


In [0]:
-- Daily transaction volume over time
-- Helps spot trends, seasonality, or data gaps

SELECT 
  transaction_date,
  COUNT(DISTINCT transaction_id) AS num_transactions,
  COUNT(DISTINCT store_id) AS num_stores_active,
  COUNT(*) AS num_line_items,
  SUM(line_total) AS total_sales
FROM workspace.silver.sales
GROUP BY transaction_date
ORDER BY transaction_date

transaction_date,num_transactions,num_stores_active,num_line_items,total_sales
2025-01-01,2051,21,6616,1590502.3799999985
2025-01-02,1998,21,6257,1497368.8299999998
2025-01-03,2121,21,6752,1568779.0599999996
2025-01-04,2369,21,8810,2112133.630000001
2025-01-05,2432,21,8994,2279838.9800000004
2025-01-06,2119,21,6877,1555671.47
2025-01-07,2013,21,6518,1559690.2000000007
2025-01-08,2065,21,6663,1631001.8399999994
2025-01-09,2064,21,6674,1583745.2500000002
2025-01-10,2062,21,6501,1529284.5200000012


## 6. Data Quality Checks

### What We're Checking:
- **NULL values** in critical columns
- **Duplicate records** (should not exist after our Silver layer processing)
- **Referential integrity** (do all IDs link properly?)

### Why This Matters:
- **NULLs** can break calculations and aggregations
- **Duplicates** inflate metrics and skew analysis
- **Orphaned records** indicate join issues
- Finding issues NOW prevents wrong conclusions LATER

### What to Look For:
- Zero nulls in key fields (IDs, dates, amounts)
- Zero duplicate line_item_ids
- All transactions have valid customer, store, product references

In [0]:
-- Check for NULL values in critical columns
-- ANY nulls in these fields could cause issues

WITH counts AS (
  SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(line_item_id)     AS line_item_id,
    COUNT(*) - COUNT(transaction_id)   AS transaction_id,
    COUNT(*) - COUNT(transaction_date) AS transaction_date,
    COUNT(*) - COUNT(customer_id)      AS customer_id,
    COUNT(*) - COUNT(store_id)         AS store_id,
    COUNT(*) - COUNT(sku_id)           AS sku_id,
    COUNT(*) - COUNT(product_id)       AS product_id,
    COUNT(*) - COUNT(category_id)      AS category_id,
    COUNT(*) - COUNT(department_id)    AS department_id,
    COUNT(*) - COUNT(quantity)         AS quantity,
    COUNT(*) - COUNT(line_total)       AS line_total
  FROM workspace.silver.sales
)
SELECT 
  column_name,
  null_count,
  ROUND(100.0 * null_count / total_rows, 2) AS null_pct
FROM counts
UNPIVOT (
  null_count FOR column_name IN (
    line_item_id,
    transaction_id,
    transaction_date,
    customer_id,
    store_id,
    sku_id,
    product_id,
    category_id,
    department_id,
    quantity,
    line_total
  )
)
ORDER BY null_count DESC;

column_name,null_count,null_pct
customer_id,2157045,44.50
line_item_id,0,0.00
transaction_id,0,0.00
transaction_date,0,0.00
store_id,0,0.00
sku_id,0,0.00
product_id,0,0.00
category_id,0,0.00
department_id,0,0.00
quantity,0,0.00


In [0]:
-- Check for duplicate line_item_ids
-- Each line item should be unique

WITH table_a as ( SELECT COUNT(*) as Total_record, COUNT(DISTINCT line_item_id) as unique_record,
COUNT(*)- COUNT(DISTINCT line_item_id) as duplicate FROM workspace.silver.sales)
SELECT field_name, field_value
FROM table_a
UNPIVOT (field_value for field_name in (total_record, unique_record, duplicate));



field_name,field_value
total_record,4846789
unique_record,4846789
duplicate,0


In [0]:
-- Check NULL values in optional/demographic fields
-- Some NULLs may be acceptable here (e.g., anonymous customers)
WITH tableA as  ( SELECT  COUNT(*) as total_record,
SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) as gender,
SUM(CASE WHEN age_group IS NULL THEN 1 ELSE 0 END) as age_group,
SUM(CASE WHEN locality IS NULL THEN 1 ELSE 0 END) as locality,
SUM(CASE WHEN loyalty_tier IS NULL THEN 1 ELSE 0 END) as loyalty_tier FROM workspace.silver.sales)
SELECT field_name , null_count, ROUND(100*null_count/total_record,2) as null_percentage
FROM tablea UNPIVOT (null_count for field_name in (gender,age_group,locality,loyalty_tier))
ORDER BY null_percentage DESC;

field_name,null_count,null_percentage
gender,2945336,60.77
age_group,2945336,60.77
locality,2945336,60.77
loyalty_tier,2157045,44.5


### Finding: NULL Values in Material Hierarchy

**Observation:** We found ~8.58% of records have NULL values in `category_id` and `department_id`.

**What does this mean?**
- These are 454,598 line items that couldn't be mapped to the material hierarchy
- The SKU and product exist, but the category/department link is missing

**Why this matters:**
- Reports grouped by department/category will exclude these records
- May need data remediation or default category assignment
- Should investigate with source system team

**Action Items for Real-World Scenario:**
1. Identify which SKUs/products are affected
2. Check if pattern exists (new products? specific brands?)
3. Work with merchandising team to assign proper categories
4. Document as known limitation for analysis

**For this learning exercise:** We'll continue analysis, noting that ~8.6% of transactions lack full hierarchy.

In [0]:
-- Investigate which products have NULL category/department
-- This helps us understand the scope of the data quality issue

SELECT 
  sku_id,
  sku_name,
  product_id,
  product_name,
  brand,
  category_id,
  department_id,
  COUNT(*) AS num_line_items
FROM workspace.silver.sales
WHERE category_id IS NULL OR department_id IS NULL
GROUP BY sku_id, sku_name, product_id, product_name, brand, category_id, department_id
ORDER BY num_line_items DESC
LIMIT 20

sku_id,sku_name,product_id,product_name,brand,category_id,department_id,num_line_items


### Critical Finding: Complete Missing Master Data

**Deeper Investigation Reveals:**
- The NULL values aren't just missing category/department
- These SKUs (SKU-090xxx series) have **NO master data at all**
  - No SKU name
  - No product name
  - No brand
  - No category or department

**Root Cause:**
- These 186 SKUs sold in transactions but don't exist in the SKU dimension table
- This is a **referential integrity issue** from the source system
- The LEFT JOIN in our Silver layer preserved these orphaned transactions

**Impact:**
- ~8.6% of line items can't be analyzed by product attributes
- Revenue is captured, but we don't know what was sold

**Real-World Actions:**
1. Report to data governance team
2. Identify if these are deleted/discontinued SKUs
3. Decide: filter out or create "Unknown" category?
4. Fix source system data quality

**Learning Point:**
> This is why EDA is critical! We discovered a major data quality issue **before** building reports. 
> Without this check, we might have built dashboards with mysteriously missing products.

## 7. Value Ranges & Outliers

### What We're Checking:
- Min/Max/Average values for numeric fields
- **Quantity**: How many items per line?
- **Amount**: Price per item range
- **Line Total**: Total per line item range
- **Discount**: Discount percentage range
- **MRP**: Maximum Retail Price range

### Why This Matters:
- **Outliers** can indicate data errors (e.g., negative quantities, impossible prices)
- **Ranges** help validate business rules
- **Averages** provide baseline expectations

### What to Look For:
- Any negative values (should not exist)
- Unrealistic maximums (e.g., 1000% discount)
- Zero values where they shouldn't be

In [0]:
-- Statistical summary of numeric fields
-- Helps identify outliers and validate ranges

SELECT 
  'quantity' AS field_name,
  MIN(quantity) AS min_value,
  ROUND(AVG(quantity), 2) AS avg_value,
  PERCENTILE(quantity, 0.5) AS median_value,
  MAX(quantity) AS max_value
FROM workspace.silver.sales

UNION ALL

SELECT 'amount', MIN(amount), ROUND(AVG(amount), 2), 
       PERCENTILE(amount, 0.5), MAX(amount)
FROM workspace.silver.sales

UNION ALL

SELECT 'mrp', MIN(mrp), ROUND(AVG(mrp), 2),
       PERCENTILE(mrp, 0.5), MAX(mrp)
FROM workspace.silver.sales

UNION ALL

SELECT 'discount_pct', MIN(discount_pct), ROUND(AVG(discount_pct), 4),
       PERCENTILE(discount_pct, 0.5), MAX(discount_pct)
FROM workspace.silver.sales

UNION ALL

SELECT 'line_total', MIN(line_total), ROUND(AVG(line_total), 2),
       PERCENTILE(line_total, 0.5), MAX(line_total)
FROM workspace.silver.sales

ORDER BY field_name

field_name,min_value,avg_value,median_value,max_value
amount,12.62,195.18,113.78,4208.59
discount_pct,0.005,0.0317,0.0251,0.2966
line_total,12.62,233.11,137.01,8365.1
mrp,14.0,203.6,117.0,4317.0
quantity,1.0,1.22,1.0,4.0


In [0]:
-- Transaction-level aggregations
-- Understanding typical basket characteristics

SELECT 
  MIN(total_amount) AS min_transaction_amount,
  ROUND(AVG(total_amount), 2) AS avg_transaction_amount,
  PERCENTILE(total_amount, 0.5) AS median_transaction_amount,
  MAX(total_amount) AS max_transaction_amount,
  MIN(n_line_items) AS min_items_per_transaction,
  ROUND(AVG(n_line_items), 2) AS avg_items_per_transaction,
  MAX(n_line_items) AS max_items_per_transaction
FROM (
  SELECT DISTINCT
    transaction_id,
    total_amount,
    n_line_items
  FROM workspace.silver.sales
)

min_transaction_amount,avg_transaction_amount,median_transaction_amount,max_transaction_amount,min_items_per_transaction,avg_items_per_transaction,max_items_per_transaction
12.95,988.09,579.66,25609.69,1,3.85,22


## 8. Distribution Patterns

### What We're Checking:
- Customer behavior patterns
  - Payment method preferences
  - Loyalty program participation
- Transaction characteristics
  - Store format distribution
  - Store tier distribution

### Why This Matters:
- Understand **customer preferences** and behavior
- Identify **business patterns** (e.g., cash vs digital payments)
- Segment analysis preparation

### What to Look For:
- Dominant payment methods
- Loyalty program adoption rate
- Store format performance

In [0]:
-- Payment method breakdown
-- Shows customer payment preferences

SELECT 
  payment_method,
  COUNT(DISTINCT transaction_id) AS num_transactions,
  ROUND(100.0 * COUNT(DISTINCT transaction_id) / SUM(COUNT(DISTINCT transaction_id)) OVER (), 2) AS pct_transactions,
  COUNT(*) AS num_line_items
FROM workspace.silver.sales
GROUP BY payment_method
ORDER BY num_transactions DESC

payment_method,num_transactions,pct_transactions,num_line_items
UPI,632648,46.01,2248547
Cash,526611,38.30,1819222
Card,215846,15.70,779020


In [0]:
-- Loyalty program participation
-- How many customers are enrolled?

SELECT 
  is_loyalty,
  COUNT(DISTINCT transaction_id) AS num_transactions,
  ROUND(100.0 * COUNT(DISTINCT transaction_id) / SUM(COUNT(DISTINCT transaction_id)) OVER (), 2) AS pct_transactions,
  COUNT(DISTINCT customer_id) AS num_customers
FROM workspace.silver.sales
GROUP BY is_loyalty
ORDER BY is_loyalty DESC

is_loyalty,num_transactions,pct_transactions,num_customers
true,539390,39.23,14359
false,835715,60.77,0


In [0]:
-- Store format and tier distribution
-- Understanding our store portfolio

SELECT 
  format AS store_format,
  tier AS store_tier,
  COUNT(DISTINCT store_id) AS num_stores,
  COUNT(DISTINCT transaction_id) AS num_transactions,
  COUNT(*) AS num_line_items,
  ROUND(SUM(line_total) / 1000000, 2) AS total_sales_millions
FROM workspace.silver.sales
GROUP BY format, tier
ORDER BY format, tier

store_format,store_tier,num_stores,num_transactions,num_line_items,total_sales_millions
Express,2,3,156465,459451,83.55
Express,3,4,199019,573119,103.74
Fresh,2,4,195446,727460,136.01
Supermarket,1,5,435390,1663368,436.65
Supermarket,2,5,388785,1423391,369.89


In [0]:
-- Top 20 brands by volume
-- Which brands dominate our sales?

SELECT 
  brand,
  COUNT(DISTINCT sku_id) AS num_skus,
  COUNT(*) AS num_line_items,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_line_items,
  SUM(CASE WHEN is_private_label THEN 1 ELSE 0 END) AS private_label_count
FROM workspace.silver.sales
GROUP BY brand
ORDER BY num_line_items DESC
LIMIT 20

brand,num_skus,num_line_items,pct_line_items,private_label_count
Amul,92,801826,16.54,0
FreshMart,115,412019,8.50,412019
Tata,59,339328,7.00,0
Ashirvaad,23,144207,2.98,0
Brooke Bond,15,141228,2.91,0
Mother Dairy,18,136645,2.82,0
Coca-Cola,20,125631,2.59,0
Pillsbury,15,114595,2.36,0
Madhur,18,108806,2.24,0
Bru,12,107791,2.22,0


In [0]:
-- Verify customer_id patterns before cleaning
-- This confirms the business rule: ANON = anonymous, CUST = identified customers

SELECT 
  CASE 
    WHEN customer_id LIKE 'ANON%' THEN 'ANON (Anonymous)'
    WHEN customer_id LIKE 'CUST%' THEN 'CUST (Identified)'
    ELSE 'Other'
  END AS customer_type,
  is_loyalty,
  COUNT(*) AS num_records,
  COUNT(DISTINCT customer_id) AS unique_customers,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM workspace.silver.sales
GROUP BY 
  CASE 
    WHEN customer_id LIKE 'ANON%' THEN 'ANON (Anonymous)'
    WHEN customer_id LIKE 'CUST%' THEN 'CUST (Identified)'
    ELSE 'Other'
  END,
  is_loyalty
ORDER BY num_records DESC

customer_type,is_loyalty,num_records,unique_customers,pct_of_total
CUST (Identified),true,2689744,14359,55.50
Other,false,2157045,0,44.50


## 9. Data Cleaning - Anonymous Customer IDs

### What We're Checking:
- Customer ID patterns: ANON vs CUST prefixes
- Relationship between customer_id patterns and loyalty program
- Business rule for handling anonymous transactions

### Why This Matters:
- **Privacy & Encryption**: ANON customer IDs represent encrypted/anonymized transactions
- **Business Logic**: These are not trackable customers - they should be NULL for proper analytics
- **Data Consistency**: Analytics should distinguish between identified customers (CUST-xxx) and anonymous transactions (NULL)

### What to Look For:
- Verify ANON pattern appears only in non-loyalty transactions
- Confirm CUST pattern appears only in loyalty transactions
- Understand impact of converting ANON to NULL

### Tables to Update:
- **transactions** - Source fact table with customer_id
- **sales** - Consolidated table inheriting customer_id from transactions

### Finding: ANON Customer IDs Need to be NULL

**Observation:**
- All non-loyalty transactions (44.2% of data) have ANON customer IDs
- All loyalty transactions have CUST customer IDs
- ANON represents encrypted/anonymous customers who cannot be tracked

**Business Rule:**
For proper analytics, ANON customer IDs should be converted to NULL because:
1. **Not Real Customers**: ANON IDs are system-generated placeholders, not actual customer identifiers
2. **Privacy Compliance**: Represents encrypted transactions for privacy/regulatory reasons
3. **Analytics Clarity**: NULL correctly indicates "no customer information available"
4. **Prevents False Aggregations**: Counting ANON-xxx as unique customers inflates customer counts

**Action:** We'll update both fact tables to set ANON customer IDs to NULL:
1. **transactions** - Source fact table
2. **sales** - Consolidated table (inherits from transactions)

In [0]:
-- Update transactions table: Convert ANON customer IDs to NULL
-- This maintains consistency across all fact tables
-- Same business rule as sales table

UPDATE workspace.silver.transactions
SET customer_id = NULL
WHERE customer_id LIKE 'ANON%';

-- Show impact
SELECT 
  'Transactions Updated' AS action,
  COUNT(*) AS num_records
FROM workspace.silver.transactions
WHERE customer_id IS NULL AND is_loyalty = false

In [0]:
-- Delete line items where product_id doesn't exist in sku_master
-- These are orphaned records from referential integrity issues
-- Line items table doesn't have product_name, so we check against sku_master

DELETE FROM workspace.silver.line_items
WHERE product_id NOT IN (SELECT DISTINCT product_id FROM workspace.silver.sku_master);

-- Show impact
SELECT 
  'Line Items Deleted' AS action,
  COUNT(*) AS remaining_records,
  COUNT(DISTINCT product_id) AS valid_products
FROM workspace.silver.line_items

In [0]:
-- Update existing table: Convert ANON customer IDs to NULL
-- This represents the business logic that anonymous transactions should not have customer IDs
-- We're modifying the table in-place rather than creating a new one

UPDATE workspace.silver.sales
SET customer_id = NULL
WHERE customer_id LIKE 'ANON%';

-- Show impact of cleaning
SELECT 
  'Records Updated' AS action,
  COUNT(*) AS num_records
FROM workspace.silver.sales
WHERE customer_id IS NULL AND is_loyalty = false

## 10. Data Cleaning - Remove Invalid Product Records

### What We're Checking:
- **line_items**: Records where product_id doesn't exist in sku_master (orphaned references)
- **sales**: Records with NULL product_name (completely invalid transactions)
- These represent garbage data from referential integrity issues

### Why This Matters:
- **Data Integrity**: Transactions without product information are meaningless
- **Analytics Quality**: Cannot analyze sales without knowing what was sold
- **System Health**: Presence of such records indicates upstream data quality issues

### Business Decision:
- Records with orphaned product_ids or NULL product_name should be **deleted**
- These are not salvageable - no way to determine what product was sold
- Better to exclude than include garbage data in reports

### Tables to Update:
- **line_items** - Source fact table (check product_id exists in sku_master)
- **sales** - Consolidated table (check product_name is not NULL)

### What to Look For:
- How many line_items have orphaned product_ids?
- How many sales records have NULL product_name?

In [0]:
-- Check sales table for records with NULL product_name
-- These are invalid transactions that should be removed

SELECT 
  'Records with NULL product_id' AS check_type,
  COUNT(*) AS num_records,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.silver.sales), 2) AS pct_of_total
FROM workspace.silver.sales
WHERE product_name IS NULL

UNION ALL

SELECT 
  'Valid records (product_id present)',
  COUNT(*),
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.silver.sales), 2)
FROM workspace.silver.sales
WHERE product_name IS NOT NULL

check_type,num_records,pct_of_total
Records with NULL product_id,0,0.00
Valid records (product_id present),4846789,100.00


In [0]:
-- Delete sales records with NULL product_name (if any exist)
-- This eliminates garbage data from our analytical dataset

DELETE FROM workspace.silver.sales
WHERE product_name IS NULL;

-- Show final record count after all cleaning
SELECT 
  'Final Record Count After Cleaning' AS status,
  COUNT(*) AS total_records,
  COUNT(DISTINCT transaction_id) AS unique_transactions,
  COUNT(CASE WHEN customer_id IS NULL THEN 1 END) AS anonymous_records,
  COUNT(CASE WHEN customer_id IS NOT NULL THEN 1 END) AS identified_customer_records
FROM workspace.silver.sales

In [0]:
-- Verify final record counts across all three fact tables
-- Ensures data cleaning was applied consistently

SELECT 'transactions' AS table_name,
       COUNT(*) AS total_records,
       COUNT(DISTINCT transaction_id) AS unique_transactions,
       COUNT(CASE WHEN customer_id IS NULL THEN 1 END) AS null_customer_ids,
       COUNT(DISTINCT CASE WHEN customer_id IS NOT NULL THEN customer_id END) AS unique_customers
FROM workspace.silver.transactions

UNION ALL

SELECT 'line_items',
       COUNT(*),
       COUNT(DISTINCT transaction_id),
       NULL AS null_customer_ids,
       COUNT(DISTINCT product_id) AS valid_products
FROM workspace.silver.line_items

UNION ALL

SELECT 'sales',
       COUNT(*),
       COUNT(DISTINCT transaction_id),
       COUNT(CASE WHEN customer_id IS NULL THEN 1 END),
       COUNT(DISTINCT CASE WHEN customer_id IS NOT NULL THEN customer_id END)
FROM workspace.silver.sales

table_name,total_records,unique_transactions,null_customer_ids,unique_customers
transactions,1380771,1380771,838318,14359
line_items,4846789,1375105,null,591
sales,4846789,1375105,2157045,14359


### Finding: Data Cleaning Summary

**Cleaning Steps Applied:**

1. **ANON Customer IDs Converted to NULL**
   - Tables Updated: **transactions** and **sales**
   - Rationale: These represent anonymous/encrypted transactions, not trackable customers
   - Impact:
     * transactions: 838,318 records updated (60.7% of transactions)
     * sales: 2,157,045 records updated (44.5% of line items)
   - Business Rule: Privacy-compliant treatment of non-loyalty transactions

2. **Orphaned Product Records Removed**
   - **line_items**: Deleted 454,598 records where product_id doesn't exist in sku_master (orphaned references)
   - **sales**: Deleted 454,598 records with NULL product_name (complete missing product data)
   - Rationale: Transactions without valid product information are garbage data
   - Impact: Removed 8.6% of records (454,598 out of 5,301,387)
   - Result: 4,846,789 clean records remain with valid product references

**Result:**
All three fact tables have been cleaned in-place and are now aligned:
- **transactions**: 1,380,771 records (838,318 anonymous, 14,359 unique customers)
- **line_items**: 4,846,789 records (591 valid products)
- **sales**: 4,846,789 records (2,157,045 anonymous, 14,359 unique customers)

Data is now ready for analysis with:
- Proper NULL handling for anonymous customers
- Guaranteed valid product references (all orphaned records removed)
- Business rules applied consistently across all tables
- Perfect alignment: line_items and sales have matching record counts

**Note for Students:**
This demonstrates a critical step in data preparation:
- **Understanding business context** (what ANON really means)
- **Applying business rules** (converting to NULL)
- **Data quality gates** (removing invalid records)
- Always document WHY you're changing data, not just WHAT you changed

## EDA Summary & Next Steps

### What We Accomplished
We systematically explored the **silver.sales** table across 10 dimensions and applied data cleaning:

1. **Table Overview**
   - Confirmed record counts and time coverage
   - Validated unique identifiers

2. **Dimensional Coverage**
   - Counted stores, departments, categories, products, SKUs, customers
   - Confirmed dimension richness

3. **Material Hierarchy**
   - Analyzed department → category → product → SKU structure
   - Identified top categories by volume

4. **Geographic Distribution**
   - Mapped regional coverage
   - Listed all stores with location details

5. **Time Period Analysis**
   - Checked date ranges per store
   - Examined daily transaction patterns

6. **Data Quality**
   - Verified no critical NULLs
   - Confirmed no duplicates
   - Checked optional field completeness
   - Identified 8.6% of records with missing material hierarchy

7. **Value Ranges**
   - Validated min/max/avg for all numeric fields
   - Identified outlier boundaries

8. **Distribution Patterns**
   - Analyzed payment methods, loyalty participation
   - Examined store formats and top brands

9. **Data Cleaning - Anonymous Customers**
   - Identified ANON customer IDs
   - Applied business rule: Updated customer_id to NULL for ANON records (privacy/encryption)
   - Tables Updated:
     * **transactions**: 838,318 records (60.7%)
     * **sales**: 2,157,045 records (44.5%)
   - Distinction: CUST = identified customers, NULL = anonymous transactions

10. **Data Cleaning - Invalid Records**
    - **line_items**: Deleted 454,598 records with orphaned product_ids (not in sku_master)
    - **sales**: Deleted 454,598 records with NULL product_name (missing product data)
    - Impact: Removed 8.6% of original records (454,598 out of 5,301,387)
    - Final state: 4,846,789 clean records across line_items and sales tables
    - All three fact tables now cleaned and aligned

---

### Key Learnings for Students

**Always explore before analyzing**
- Never jump straight to metrics and KPIs
- Understanding data structure prevents errors

**Follow a systematic approach**
- Coverage → Quality → Ranges → Distributions
- Document findings as you go

**Look for red flags**
- Missing dates = incomplete data
- Unexpected NULLs = join issues
- Extreme outliers = data errors

**Build context before insights**
- Know what you have before asking what it means
- Context makes analysis meaningful

---

### Ready for Next Steps
Now that we **understand our data** and **applied necessary cleaning**, we can confidently:
- Use **workspace.silver.sales** for all downstream analysis (cleaned in-place)
- All three fact tables (**transactions**, **line_items**, **sales**) are consistent and clean
- Build business metrics (sales, growth, margins)
- Create aggregations (daily, weekly, monthly)
- Perform segment analysis (by region, category, customer)
- Design dashboards and reports

**Critical Learning:**
We didn't just explore - we also **fixed** data issues across all fact tables:
- Applied business rules consistently (ANON → NULL in transactions and sales)
- Removed invalid data (orphaned product_ids from line_items, NULL product_name from sales)
- Maintained referential integrity across all tables
- Documented WHY we made changes
- Created clean, analysis-ready datasets